# Weather Documents → Vector Embeddings (Lakebase)

## Overview

This Part 2 notebook reads existing `weather_documents`, chunks `narrative_text`, generates MiniLM embeddings, and writes them directly to a Lakebase `vector(384)` column. It does not fetch weather, use Spark, expose search, create query embeddings, or implement RAG.

The pipeline is model-aware and idempotent: a document is selected only when it has no `weather_embeddings` row for the configured model. A successful second run therefore has no work to perform.


## Install/import dependencies

Databricks notebook compute does not inherit the FastAPI environment. `databricks-sdk` is required by `app.database`, while `pydantic-settings` is required by `app.config`. `psycopg2-binary` supplies the driver used by `get_connection()`, and `sentence-transformers` supplies the embedding model.


In [0]:
%pip uninstall -y psycopg2 psycopg2-binary
%pip install -q sentence-transformers 'databricks-sdk>=0.118.0'


In [0]:
dbutils.library.restartPython()


## Configuration

The widgets below populate only the environment variables already consumed by `app.config.Settings` and `app.database.get_connection()`. They are not a separate connection implementation. OAuth credentials continue to be generated by the Databricks SDK inside `get_connection()`.


In [0]:
import os

from databricks.sdk import WorkspaceClient

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBEDDING_BATCH_SIZE = 32
UPSERT_PAGE_SIZE = 100

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")

default_pg_user = os.getenv("PGUSER", "").strip()
if not default_pg_user:
    try:
        default_pg_user = WorkspaceClient().current_user.me().user_name or ""
    except Exception:
        default_pg_user = ""

dbutils.widgets.text("pg_host", os.getenv("PGHOST", "ep-summer-smoke-d8pmbi3s.database.us-east-2.cloud.databricks.com"), "Lakebase host")
dbutils.widgets.text("pg_database", os.getenv("PGDATABASE", "databricks_postgres"), "Lakebase database")
dbutils.widgets.text("pg_user", default_pg_user, "Lakebase OAuth user")
dbutils.widgets.text("pg_port", os.getenv("PGPORT", "5432"), "Lakebase port")
dbutils.widgets.text("pg_sslmode", os.getenv("PGSSLMODE", "require"), "Postgres SSL mode")
dbutils.widgets.text("endpoint_name", os.getenv("ENDPOINT_NAME", "projects/weather-retrieval-system/branches/production/endpoints/primary"), "Lakebase endpoint resource name")

print("Lakebase widgets created. Enter pg_host and endpoint_name above, then run the next cell.")
print(f"Available widgets: {sorted(dbutils.widgets.getAll())}")


### Apply and validate connection settings

After the widget-creation cell completes, enter `pg_host` and the full `endpoint_name` in the widget panel. Then run this cell to copy those values into the existing application environment contract.


In [0]:
connection_environment = {
    "APP_ENV": "databricks",
    "PGHOST": dbutils.widgets.get("pg_host").strip(),
    "PGDATABASE": dbutils.widgets.get("pg_database").strip(),
    "PGUSER": dbutils.widgets.get("pg_user").strip(),
    "PGPORT": dbutils.widgets.get("pg_port").strip(),
    "PGSSLMODE": dbutils.widgets.get("pg_sslmode").strip(),
    "ENDPOINT_NAME": dbutils.widgets.get("endpoint_name").strip(),
}

missing_settings = [
    name
    for name in ("PGHOST", "PGDATABASE", "PGUSER", "ENDPOINT_NAME")
    if not connection_environment[name]
]
if missing_settings:
    raise ValueError(
        "Complete the Lakebase widgets before continuing: "
        + ", ".join(missing_settings)
    )

if connection_environment["PGSSLMODE"] not in {"require", "verify-ca", "verify-full"}:
    raise ValueError("PGSSLMODE must be require, verify-ca, or verify-full")
try:
    pg_port = int(connection_environment["PGPORT"])
except ValueError as exc:
    raise ValueError("PGPORT must be an integer") from exc
if not 1 <= pg_port <= 65535:
    raise ValueError("PGPORT must be between 1 and 65535")

os.environ.update(connection_environment)

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Chunk size/overlap: {CHUNK_SIZE}/{CHUNK_OVERLAP}")
print(f"Lakebase host: {connection_environment['PGHOST']}")
print(f"Lakebase database/user: {connection_environment['PGDATABASE']} / {connection_environment['PGUSER']}")

## Create the Lakebase connection context manager

Keep the notebook independent from the FastAPI runtime. This cell mirrors the application's OAuth connection behavior without importing workspace files: generate a fresh Lakebase credential, open a direct psycopg2 connection, and always close it.

In [0]:
from collections.abc import Iterator
from contextlib import contextmanager

import psycopg2
from databricks.sdk import WorkspaceClient
from psycopg2.extensions import connection as PsycopgConnection
from psycopg2.extras import RealDictCursor

workspace_client = WorkspaceClient()


@contextmanager
def get_connection() -> Iterator[PsycopgConnection]:
    credential = workspace_client.postgres.generate_database_credential(
        endpoint=connection_environment["ENDPOINT_NAME"]
    )
    if not credential.token:
        raise RuntimeError("Databricks did not return a Lakebase OAuth token")

    connection = psycopg2.connect(
        host=connection_environment["PGHOST"],
        port=pg_port,
        dbname=connection_environment["PGDATABASE"],
        user=connection_environment["PGUSER"],
        password=credential.token,
        sslmode=connection_environment["PGSSLMODE"],
        connect_timeout=15,
        cursor_factory=RealDictCursor,
    )
    try:
        yield connection
    finally:
        connection.close()


print("Notebook-local Lakebase OAuth connection helper is ready")


## Verify Lakebase connection and pgvector

The assignment states that pgvector is already enabled. This cell verifies it and fails clearly when it is unavailable; it does not create or reconfigure the extension.


In [0]:
with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT current_database() AS database_name, current_user AS database_user"
        )
        connection_info = cursor.fetchone()
        cursor.execute("SELECT COUNT(*) AS document_count FROM weather_documents")
        source_count = cursor.fetchone()["document_count"]
        cursor.execute(
            "SELECT extversion FROM pg_extension WHERE extname = 'vector'"
        )
        vector_extension = cursor.fetchone()

if vector_extension is None:
    raise RuntimeError("pgvector is not enabled in this Lakebase database")

print(
    f"Connected to {connection_info['database_name']} as {connection_info['database_user']}"
)
print(f"weather_documents rows: {source_count}")
print(f"pgvector version: {vector_extension['extversion']}")


## Ensure `weather_embeddings` schema and HNSW index

The destination column is created as `vector(384)` from the outset. No intermediate array column or cast-after-insert migration is used.


In [0]:
WEATHER_EMBEDDINGS_DDL = """
CREATE TABLE IF NOT EXISTS weather_embeddings (
    id TEXT PRIMARY KEY,
    document_id TEXT NOT NULL
        REFERENCES weather_documents(id) ON DELETE CASCADE,
    chunk_index INTEGER NOT NULL,
    chunk_text TEXT NOT NULL,
    embedding vector(384) NOT NULL,
    model_name TEXT NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    UNIQUE (document_id, chunk_index, model_name)
)
"""

WEATHER_EMBEDDINGS_HNSW_INDEX = """
CREATE INDEX IF NOT EXISTS weather_embeddings_embedding_hnsw_idx
ON weather_embeddings
USING hnsw (embedding vector_cosine_ops)
"""

with get_connection() as connection:
    try:
        with connection.cursor() as cursor:
            cursor.execute(WEATHER_EMBEDDINGS_DDL)
            cursor.execute(WEATHER_EMBEDDINGS_HNSW_INDEX)
        connection.commit()
    except Exception:
        connection.rollback()
        raise

print("weather_embeddings table and HNSW cosine index are ready")


## Load unembedded weather documents

Only nonblank source documents with no embedding row for this model are selected. This deliberately simple Part 2 rule does not detect later narrative changes when the document ID remains unchanged.


In [0]:
UNEMBEDDED_DOCUMENTS_SQL = """
SELECT d.id, d.narrative_text
FROM weather_documents AS d
WHERE BTRIM(d.narrative_text) <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM weather_embeddings AS e
      WHERE e.document_id = d.id
        AND e.model_name = %s
  )
ORDER BY d.id
"""

with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(UNEMBEDDED_DOCUMENTS_SQL, (EMBEDDING_MODEL_NAME,))
        source_documents = list(cursor.fetchall())

print(f"Source documents selected: {len(source_documents)}")
if source_documents:
    display(source_documents[:5])


## Chunk `narrative_text`

Chunking uses an overlapping character window. Short weather narratives naturally produce one chunk.


In [0]:
def chunk_narrative(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")
    if chunk_size <= 0 or chunk_overlap < 0:
        raise ValueError("chunk_size must be positive and chunk_overlap non-negative")

    normalized_text = text.strip()
    if not normalized_text:
        return []

    step = chunk_size - chunk_overlap
    chunks = []
    for start in range(0, len(normalized_text), step):
        chunk = normalized_text[start : start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(normalized_text):
            break
    return chunks


assert chunk_narrative("") == []
assert chunk_narrative("short forecast") == ["short forecast"]
assert len(chunk_narrative("x" * CHUNK_SIZE)) == 1
overlap_test_chunks = chunk_narrative("x" * (CHUNK_SIZE + 1))
assert [len(chunk) for chunk in overlap_test_chunks] == [CHUNK_SIZE, CHUNK_OVERLAP + 1]

chunk_records = []
for document in source_documents:
    document_chunks = chunk_narrative(document["narrative_text"])
    for chunk_index, chunk_text in enumerate(document_chunks):
        chunk_records.append(
            {
                "document_id": document["id"],
                "chunk_index": chunk_index,
                "chunk_text": chunk_text,
            }
        )

print(f"Chunks generated: {len(chunk_records)}")
if chunk_records:
    display(chunk_records[:5])


## Load the embedding model

The model is loaded once, only when chunks require embedding. Its declared output dimension must match the Lakebase column.


In [0]:
from sentence_transformers import SentenceTransformer

model = None
if chunk_records:
    huggingface_cache = os.environ.setdefault(
        "HF_HOME", "/tmp/.cache/huggingface"
    )
    print(f"Loading {EMBEDDING_MODEL_NAME}...")
    model = SentenceTransformer(
        EMBEDDING_MODEL_NAME, cache_folder=huggingface_cache
    )
    model_dimension = model.get_sentence_embedding_dimension()
    if model_dimension != EMBEDDING_DIM:
        raise ValueError(
            f"Model dimension {model_dimension} does not match {EMBEDDING_DIM}"
        )
    print(f"Loaded one {EMBEDDING_DIM}-dimensional embedding model")
else:
    print("No chunks require embedding; model loading skipped")


## Generate embeddings

Chunks are encoded synchronously in batches of 32. Every generated vector is checked for dimension and finite numeric values before database serialization.


In [0]:
import hashlib
import math


def deterministic_embedding_id(
    model_name: str, document_id: str, chunk_index: int
) -> str:
    identity = f"{model_name}|{document_id}|{chunk_index}"
    digest = hashlib.sha256(identity.encode("utf-8")).hexdigest()
    return f"weather_embedding:{digest}"


def pgvector_literal(vector: list[float]) -> str:
    if len(vector) != EMBEDDING_DIM:
        raise ValueError(
            f"Expected {EMBEDDING_DIM} values, received {len(vector)}"
        )
    if not all(math.isfinite(value) for value in vector):
        raise ValueError("Embedding contains a non-finite value")
    return "[" + ",".join(format(value, ".9g") for value in vector) + "]"


assert deterministic_embedding_id("model", "doc", 0) == deterministic_embedding_id("model", "doc", 0)
assert deterministic_embedding_id("model", "doc", 0) != deterministic_embedding_id("model", "doc", 1)

embedding_vectors = []
if chunk_records:
    assert model is not None
    for start in range(0, len(chunk_records), EMBEDDING_BATCH_SIZE):
        batch_records = chunk_records[start : start + EMBEDDING_BATCH_SIZE]
        batch_vectors = model.encode(
            [record["chunk_text"] for record in batch_records],
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
        for vector in batch_vectors:
            values = [float(value) for value in vector.tolist()]
            if len(values) != EMBEDDING_DIM:
                raise ValueError(
                    f"Generated vector dimension {len(values)} does not match {EMBEDDING_DIM}"
                )
            if not all(math.isfinite(value) for value in values):
                raise ValueError("Generated embedding contains a non-finite value")
            embedding_vectors.append(values)
        print(
            f"Embedded {min(start + EMBEDDING_BATCH_SIZE, len(chunk_records))}/{len(chunk_records)} chunks"
        )

if len(embedding_vectors) != len(chunk_records):
    raise RuntimeError("Chunk and embedding counts do not match")

print(f"Embeddings generated: {len(embedding_vectors)}")


## Batch upsert embeddings

`execute_values` writes pgvector text directly through `%s::vector`. The logical uniqueness key makes reruns safe, while `DO UPDATE` refreshes a repeated logical chunk.


In [0]:
from psycopg2.extras import execute_values

WEATHER_EMBEDDINGS_UPSERT = """
INSERT INTO weather_embeddings (
    id, document_id, chunk_index, chunk_text, embedding, model_name, created_at
)
VALUES %s
ON CONFLICT (document_id, chunk_index, model_name) DO UPDATE
SET id = EXCLUDED.id,
    chunk_text = EXCLUDED.chunk_text,
    embedding = EXCLUDED.embedding,
    created_at = now()
"""

embedding_rows = [
    (
        deterministic_embedding_id(
            EMBEDDING_MODEL_NAME,
            chunk["document_id"],
            chunk["chunk_index"],
        ),
        chunk["document_id"],
        chunk["chunk_index"],
        chunk["chunk_text"],
        pgvector_literal(vector),
        EMBEDDING_MODEL_NAME,
    )
    for chunk, vector in zip(chunk_records, embedding_vectors, strict=True)
]

embeddings_written = 0
if embedding_rows:
    with get_connection() as connection:
        try:
            with connection.cursor() as cursor:
                execute_values(
                    cursor,
                    WEATHER_EMBEDDINGS_UPSERT,
                    embedding_rows,
                    template="(%s, %s, %s, %s, %s::vector, %s, now())",
                    page_size=UPSERT_PAGE_SIZE,
                )
            connection.commit()
            embeddings_written = len(embedding_rows)
        except Exception:
            connection.rollback()
            raise

print(f"Embeddings written: {embeddings_written}")


## Validation / summary queries

The summary checks source, chunk, and write counts; validates stored dimensions; shows compact sample rows; and reports totals by model.


In [0]:
with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                COUNT(*) AS embedding_count,
                COUNT(DISTINCT document_id) AS embedded_document_count,
                COUNT(*) FILTER (WHERE vector_dims(embedding) <> %s) AS invalid_dimension_count
            FROM weather_embeddings
            WHERE model_name = %s
            """,
            (EMBEDDING_DIM, EMBEDDING_MODEL_NAME),
        )
        model_summary = cursor.fetchone()

        cursor.execute(
            """
            SELECT
                id, document_id, chunk_index,
                LEFT(chunk_text, 120) AS chunk_preview,
                vector_dims(embedding) AS embedding_dimension,
                model_name, created_at
            FROM weather_embeddings
            WHERE model_name = %s
            ORDER BY created_at DESC, document_id, chunk_index
            LIMIT 5
            """,
            (EMBEDDING_MODEL_NAME,),
        )
        sample_embeddings = list(cursor.fetchall())

        cursor.execute(
            """
            SELECT
                model_name, COUNT(*) AS embedding_count,
                COUNT(DISTINCT document_id) AS document_count,
                MIN(created_at) AS first_created_at,
                MAX(created_at) AS last_created_at
            FROM weather_embeddings
            GROUP BY model_name
            ORDER BY model_name
            """
        )
        counts_by_model = list(cursor.fetchall())

if model_summary["invalid_dimension_count"] != 0:
    raise RuntimeError("Stored embeddings with an unexpected dimension were found")

print("Run summary")
print(f"  Source documents selected: {len(source_documents)}")
print(f"  Chunks generated: {len(chunk_records)}")
print(f"  Embeddings written: {embeddings_written}")
print(f"  Stored embeddings for model: {model_summary['embedding_count']}")
print(f"  Embedded documents for model: {model_summary['embedded_document_count']}")
print(f"  Invalid stored dimensions: {model_summary['invalid_dimension_count']}")

print("Sample weather_embeddings rows")
display(sample_embeddings)
print("Counts grouped by model_name")
display(counts_by_model)

if not source_documents:
    print("No unembedded documents were found; this is the expected second-run result.")
